In [110]:
import pandas as pd
import numpy as np
from EssSimulation_withoutMaxDemand import EssSimulationModel
import calendar
import copy

In [111]:
exp_name = "estimate830"
month_num = 9
node_name = "route_B_{:02d}".format(month_num)

In [112]:
es_info = {"transform_capacity": 63000,
           "invertband": 0,
           "soc_redundant_ratio": 0,
           "usable_depth": 0.97,
           "charge_loss": 0.92,
           "discharge_loss": 0.95,
           "es_charge_max": 9000,
           "es_charge_min": -9000,
           "es_capacity_max": 18000,
           "es_capacity_min": 0}

In [113]:
ratio_result_list = []
for ratio in range(100, 500, 10):
    demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/demand_load.csv")
    demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
    demand_load_df.set_index('time', inplace=True)

    strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/2stage_ideal_ratio_dod97/schedule_result_fixline_up{ratio}.csv")
    strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
    strategy_df['time'] = pd.to_datetime(strategy_df['time'])
    strategy_df.set_index('time', inplace=True)

    ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ele_price.csv")
    ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
    ele_price_df.set_index('time', inplace=True)

    simulation_model = EssSimulationModel(es_info)
    es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 4050)
    origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, 38.4)
    ori_max_demand = total_load_df["total_load"].mean() * 1.1
    opt_max_demand = total_load_df["total_load"].max()
    max_demand_lift_cost = (opt_max_demand - ori_max_demand) * 38.4
    gross_income = origin_balance - opt_balance - max_demand_lift_cost
    ratio_result_list.append((ratio, gross_income))
    print(f"突破比例{ratio}%, 收益为{gross_income}")
    # print(round(origin_balance - opt_balance,0))

突破比例100%, 收益为173480.87309871937
突破比例110%, 收益为189904.92721815794
突破比例120%, 收益为206328.9788851019
突破比例130%, 收益为222753.0401204976
突破比例140%, 收益为239483.63715601483
突破比例150%, 收益为256361.94890066207
突破比例160%, 收益为273240.18904188124
突破比例170%, 收益为289561.9189383453
突破比例180%, 收益为305220.83027130686
突破比例190%, 收益为320797.04912174755
突破比例200%, 收益为334654.07564159366
突破比例210%, 收益为344623.38920658163
突破比例220%, 收益为351927.3473559061
突破比例230%, 收益为358435.12074037944
突破比例240%, 收益为364705.32107369567
突破比例250%, 收益为369894.3777018951
突破比例260%, 收益为372250.2885934274
突破比例270%, 收益为372094.06406948564
突破比例280%, 收益为371917.0360981966
突破比例290%, 收益为371814.0927602436
突破比例300%, 收益为371711.1054484617
突破比例310%, 收益为371608.1588256575
突破比例320%, 收益为371409.3005902081
突破比例330%, 收益为371099.3622351283
突破比例340%, 收益为370751.74049675267
突破比例350%, 收益为370312.4064464566
突破比例360%, 收益为369871.779955287
突破比例370%, 收益为369431.11865457776
突破比例380%, 收益为368990.4913663561
突破比例390%, 收益为368549.82921126555
突破比例400%, 收益为368018.89285106235
突破比例410%, 收益为367318.4241

In [114]:
max_ratio_tuple = max(ratio_result_list, key=lambda x: x[1])

In [115]:
max_ratio = max_ratio_tuple[0]
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df.set_index('time', inplace=True)

strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/2stage_ideal_ratio_dod97/schedule_result_fixline_up{max_ratio}.csv")
strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
strategy_df['time'] = pd.to_datetime(strategy_df['time'])
strategy_df.set_index('time', inplace=True)

ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df.set_index('time', inplace=True)

simulation_model = EssSimulationModel(es_info)
es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 4050)
origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, 38.4)

In [116]:
ori_max_demand = total_load_df["total_load"].mean() * 1.1
opt_max_demand = total_load_df["total_load"].max()
max_demand_lift_cost = (opt_max_demand - ori_max_demand) * 38.4
gross_income = origin_balance - opt_balance - max_demand_lift_cost

print("ratio：", max_ratio,
      "调度后最大需量：", opt_max_demand,
      "原始最大需量：", ori_max_demand,
      "需量抬升成本：", max_demand_lift_cost,
      "总收益：", gross_income)

ratio： 260 调度后最大需量： 12851.93 原始最大需量： 10651.100172199984 需量抬升成本： 84511.86538752064 总收益： 372250.2885934274
